| # | Topic |
|---|-------|
| 1 | Why Vector RAG Fails on Professional Documents |
| 2 | How PageIndex Builds a Tree Index from a PDF |
| 3 | LLM Tree Search — Reasoning Over Structure |
| 4 | Full End-to-End Vectorless RAG Pipeline |
| 5 | Expert-Guided Retrieval (Domain Knowledge Injection) |
| 6 | Chat API — Zero LLM Setup |
| 7 | Self-Hosted Open-Source Option |



## 🔑 Key Concept

> **Traditional RAG** → Chunk → Embed → Cosine Similarity → Retrieve
>
> **PageIndex RAG** → Build Tree → LLM Reasons Over Tree → Retrieve Exact Sections

### The Problem with Vector RAG

**Similarity ≠ Relevance**

A chunk about "market conditions" may score higher than the actual answer section simply because it shares more words with your query.


In [1]:
import os, json, time
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")



llm = ChatOpenAI(
    model="prism-ml/ternary-bonsai-2-27b",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=1000,
)


KeyboardInterrupt: 

In [ ]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)


print("✅ PageIndex client ready")

✅ PageIndex client ready


c:\Users\Lenovo\Desktop\Agentic AI\Vectorless_RAG\.venv\Lib\site-packages\pageindex\__init__.py:9: UserWarning: You're on Python 3.13, so an older pageindex (0.2.8.1) was installed. pageindex 0.2.10+ requires Python 3.10+.
  warnings.warn(_OLDER_INSTALL + " pageindex 0.2.10+ requires Python 3.10+.")



## 🌲 Section 2: Upload & Index a PDF

### What Happens Here?

1. Upload your PDF to the PageIndex cloud.
2. PageIndex uses an LLM to analyze the document's structure.
3. Builds a hierarchical **tree index**, similar to a smart Table of Contents.
4. Returns a `doc_id` for future operations.

### Why No Chunking?

Instead of splitting the document into arbitrary 500-token chunks, PageIndex respects the document's natural section boundaries — chapters, sub-sections, and paragraphs — as intended by the author.


In [ ]:
PDF_PATH = "../data/sample_document.pdf"  

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ../data/sample_document.pdf
✅ Uploaded!
📋 Document ID: pi-cmv53vriz00040bpa06unolc9
   (Save this ID — you'll use it throughout the notebook)


In [ ]:
print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: queued
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!



## 🔍 Section 3: Inspect the Tree Structure

### What the Tree Looks Like

```text
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

### Each Node Has

- `node_id` — Unique ID used during retrieval.
- `title` — Section heading.
- `page_index` — Page number in the original PDF.
- `text` — Section summary (when `node_summary=True`).
- `nodes` — Child sections nested under the current node.

**Key takeaway:** This hierarchical tree structure is what the LLM reasons over during retrieval to identify the most relevant sections of the document.


In [ ]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 2

🌲 Raw tree (first node):
{
  "title": "Preface",
  "node_id": "0000",
  "page_index": 1,
  "start_index": 1,
  "end_index": 1,
  "summary": "ARTIFICIAL INTELLIGENCE MACHINE LEARNING AND DEEP LEARNING\n\n![img-0.jpeg](img-0.jpeg)\n\nO. CAMPESATO\n",
  "text": "ARTIFICIAL INTELLIGENCE MACHINE LEARNING AND DEEP LEARNING\n\n![img-0.jpeg](img-0.jpeg)\n\nO. CAMPESATO\n"
}


In [ ]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Preface  (p.1)
[0001] ARTIFICIAL INTELLIGENCE  (p.2)
  └─ [0002] MACHINE LEARNING  (p.2)
    └─ [0003] MACHINE LEARNING (intro)  (p.2)
    └─ [0004] LICENSE, DISCLAIMER OF LIABILITY, AND LIMITED WARRANTY  (p.3)
    └─ [0005] ARTIFICIAL INTELLIGENCE MACHINE LEARNING AND DEEP LEARNING  (p.4)
      └─ [0006] Title Page, Copyright, Dedication, and Table of Contents  (p.4)
      └─ [0007] Table of Contents  (p.8)
      └─ [0008] Table of Contents and Preface  (p.12)
    └─ [0009] PREFACE: THE ML AND DL LANDSCAPE  (p.16)
    └─ [0010] INTRODUCTION TO AI  (p.20)
      └─ [0011] INTRODUCTION TO AI (intro)  (p.20)
      └─ [0012] What Is Artificial Intelligence?  (p.21)
      └─ [0013] The Turing Test  (p.24)
      └─ [0014] Heuristics  (p.25)
      └─ [0015] Knowledge Representation  (p.27)
      └─ [0016] AI and Games  (p.29)
      └─ [0017] Expert Systems  (p.31)
      └─ [0018] Neural Computing  (p.32)
      └─ [0019] Evolutionary Computation  (p.33)
     

In [ ]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 191
   Each node = one retrievable section of the document



## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from Vector RAG.**

### Vector RAG Retrieval

```text
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```

*Problem:* Finds what's similar, not necessarily what's relevant.

### PageIndex Retrieval

```text
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```

*Advantage:* The LLM understands document structure, context, and intent to identify the most relevant sections.

### Key Takeaway

**The LLM acts like a human expert scanning a Table of Contents** to locate the sections most likely to contain the answer.


In [ ]:
def llm_tree_search(query: str, tree: list) -> dict:
    """
    Core PageIndex retrieval using OpenRouter.

    Sends the query and compressed document tree to an LLM.
    Returns relevant node IDs in JSON format.
    """

    # Compress the tree to save tokens
    def compress(nodes):
        out = []

        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }

            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])

            out.append(entry)

        return out

    compressed_tree = compress(tree)

    prompt = f"""
You are given a query and a document's tree structure,
similar to a Table of Contents.

Identify the node IDs most likely to contain the answer.
Use the section titles, summaries, and hierarchy to guide
your selection.

Query:
{query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this JSON format:
{{
    "thinking": "Brief explanation of the selected sections",
    "node_list": ["node_id1", "node_id2"]
}}
"""

    # Call the model through OpenRouter
    response = llm.invoke(prompt)

    # Parse the JSON returned by the model
    content = response.content

    if isinstance(content, list):
        content = "".join(
            item["text"] for item in content
            if isinstance(item, dict) and "text" in item
        )

    return json.loads(content)

In [ ]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What Are Word Embeddings?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What Are Word Embeddings?

🧠 LLM Reasoning:
The query matches the section title 'What Are Word Embeddings?' exactly, which is a specific node in the NLP chapter.

🎯 Selected Node IDs: ['0106']



## ⚙️ Section 5: Full End-to-End RAG Pipeline

### The 3 Steps

1. **Tree Search** → The LLM identifies relevant `node_ids` from the document tree.
2. **Retrieve** → Fetch the actual section content corresponding to the selected nodes.
3. **Generate** → The LLM generates a grounded answer with page citations.

### What Makes This Better Than Vector RAG?

- Retrieved content includes section titles and page numbers, making the sources traceable.
- The LLM can cite the specific sections and pages supporting its answer.
- Structural retrieval can reduce irrelevant context and the risk of hallucinations.

### Key Takeaway

**PageIndex RAG = Structure-Based Retrieval + Relevant Content + Grounded Answers**

The goal is to retrieve the right sections of a document rather than relying solely on similarity scores between vector embeddings.


In [ ]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [ ]:

# ── Generate Answer from Retrieved Nodes ──────────────────────────

def generate_answer(query: str, nodes: list) -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """

    if not nodes:
        return "⚠️ No relevant sections found in the document."

    # Build context string from retrieved nodes
    context_parts = []

    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | "
            f"Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""
You are an expert document analyst.

Answer the question using ONLY the provided context.

Instructions:
- Cite section titles and page numbers for factual claims.
- Be concise, clear, and precise.
- If the context does not contain the answer, explicitly say so.
- Do not invent facts or citations.

Question:
{query}

Context:
{context}

Answer:
"""

    # Call the model through OpenRouter
    response = llm.invoke(prompt)

    return response.content


In [ ]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [ ]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What are word embeddings?",
    tree=pageindex_tree
)

🔍 Query: What are word embeddings?

🧠 Reasoning: The section titled 'What Are Word Embeddings?' directly addresses the query, and its summary begins by defining an embedding as a fixed-length vector used to represent an entity such as a word....
🎯 Retrieved node IDs: ['0106']
📄 Sections found: ['What Are Word Embeddings?']

📝 Answer:


Word embeddings are fixed-length, real-valued vectors used to encode and represent linguistic entities such as words (and also documents, sentences, or graphs) in a numerical space; word vectors can be sparse, with one-hot encoding as an example. Word representations are typically learned using popular unsupervised embedding algorithms such as Word2vec, GloVe, and FastText, which rely on the distributional hypothesis: words appearing in similar contexts tend to have similar meanings.  

Citation: “What Are Word Embeddings?” (Page 187).


In [ ]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What Are Random Forests?",
    "How to Resize Images in Keras?",
    "How to Convert Python Arrays to TF Tensors?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What Are Random Forests?
A: 

Random Forests are a generalization of decision trees that use multiple decision trees. For numeric predictions, they average the trees’ predictions; for categorical predictions, they use the mode of the trees’ predictions. They operate similarly to portfolio diversification, balancing risks with ...
-------------------------------------------------------

Q: How to Resize Images in Keras?
A: 

In **“Resizing Images in Keras” (Page 223)**, resize an image by:

1. Creating a Keras input layer for a 3-channel image.
2. Applying a `Lambda` layer that calls `tf.image.resize`.
3. Building a Keras model with that input and resized output.
4. Loading an image and predicting through the model.

...
-------------------------------------------------------

Q: How to Convert Python Arrays to TF Tensors?
A: 

Use `tf.convert_to_tensor()` on a Python/NumPy array, specifying the desired dtype:

```python
import tensorflow as tf
import numpy as np

x1 = np.array([[1.


## 🎓 Section 6: Expert-Guided Retrieval

**A powerful feature of PageIndex is the ability to incorporate domain expertise directly into retrieval.**

### Why Is This Useful?

With traditional vector RAG, adapting retrieval to specialized terminology and domain-specific relationships may require improving the embedding strategy or, in some cases, fine-tuning the embedding model.

With PageIndex, you can inject expert knowledge directly into the LLM prompt using simple rules.

### Example: Domain-Specific Rules

```text
"If the query mentions EBITDA → prioritize the MD&A section."

"If the query is about risks → check Part I, Item 1A."
```

### Applications Across Different Domains

- **Finance:** Prioritize financial statements, MD&A, and risk disclosures.
- **Legal:** Search relevant clauses, statutes, and case sections.
- **Medical:** Prioritize relevant clinical guidelines and evidence sections.
- **Technical:** Focus on specifications, troubleshooting, and implementation details.

### Key Takeaway

**Domain Knowledge + Tree Search = More Targeted Retrieval**

PageIndex allows you to adapt retrieval behavior by adding expert-defined rules to the prompt, without necessarily training or fine-tuning a model.

*Note: Retrieval quality still depends on the document structure, the accuracy of the rules, and the LLM's ability to follow them.*


In [ ]:
# ── Define domain expert rules ───────────────────────────────────────────────
# These are routing rules that tell the LLM WHERE to look for specific queries.
# Think of it as encoding a senior analyst's institutional knowledge.

In [ ]:
FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

In [ ]:
# ── Expert Routing Rules — Advanced Route of Learning AI ─────────────────────
# 21 Modules | 38 Sections | 481 Topics
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
 
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
 
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""